# Clientes de Bancos

A pedido de uma instituição financeira foi solicitado que se crie um modelo de AM para fazer predições de quais clientes irão deixar abandonar a instituição, para se construir este modelo irei usar os dados dos clientes e historico de clientes que deixaram a mesma, fornecidos pela instituição.

## Inicialização

Para este projeto irei fazer uso das seguintes bibliotecas para contrução e avaliação do modelo: pandas, numpy, math, matplotlib, seaborn e sklearn.

__Carregando Bibliotecas Necessarias__

Na celula abaixo são importadas as bibliotecas que serão usadas ao longo do projeto.

In [ ]:
#Carregando as bibliotecas necessarias
import pandas as pd
import numpy as np
import math
import matplotlib.pyplot as plt
from scipy import stats as st
import seaborn as sns

#Bibliotecas de manipulação de modelos e treinamento
from sklearn.model_selection import train_test_split, GridSearchCV
from sklearn.ensemble import RandomForestClassifier
from sklearn.preprocessing import OneHotEncoder, StandardScaler
from sklearn.utils import shuffle
from sklearn.metrics import f1_score, accuracy_score, roc_auc_score

## Carregando os dados

Nesta celula abaixo serão importados os dados do arquivo Churn.csv que contem todos os dados que serão usados.

In [ ]:
dados = pd.read_csv("/datasets/Churn.csv")

## Analizando Meta Dados

Neste Capitulo iremos analizar o conjunto de dados que nos foram fornecidos pela empresa e verificar se eles estão em condições aceitaveis para se fazer analise e para se treinar modelos com eles, caso não estejam, serão modificados para cumprirem os  requisitos necessarios para se criar modelos de dados.

__Analisando a estrutura do conjunto de dados__

In [ ]:
dados.shape

__Obtendo informações do objeto pandas__

In [ ]:
dados.info()

In [ ]:
dados.describe()

__Imprimindo as 15 primeiras linhas__

In [ ]:
dados.head(15)

__Identificando os dados ausentes em nosso conjunto de dados__

In [ ]:
dados.Tenure.isna().sum()

__Preenchendo os dados ausentes encontrados no conjunto de dados__

In [ ]:
dados.Tenure = dados.Tenure.fillna(dados.Tenure.median())

__Verificando o resultado final__

In [ ]:
dados.info()

In [ ]:
dados.describe()

__Verificando os dados ausentes__

In [ ]:
dados.Tenure.isna().sum()

## Preparação dos Dados

Seria ideal dividir os dados do conjunto de dados em dados de treinamento, de testes e dados de validação final para que o nosso modelo não fique apenas treinados nestes dados não tenha muitas capabilidades com dados da vida real.

In [ ]:
dados.head(5)

__Executando a codificação OHE para a coluna Gender__

In [ ]:

encoder = OneHotEncoder(sparse_output=False)

one_hot_encoded = encoder.fit_transform(dados[['Gender']])

one_hot_encoded_dados = pd.DataFrame(one_hot_encoded, columns=encoder.get_feature_names_out(['Gender']))

dados_encoded = pd.concat([dados, one_hot_encoded_dados], axis=1)

dados_encoded.drop('Gender', axis=1, inplace=True)


__Executando a codificação OHE para a coluna Geography__

In [ ]:
one_hot_encoded = encoder.fit_transform(dados[['Geography']])

one_hot_encoded_geography = pd.DataFrame(one_hot_encoded, columns=encoder.get_feature_names_out(['Geography']))

dados_ohe = pd.concat([dados_encoded, one_hot_encoded_geography], axis=1)

dados_ohe.drop('Geography', axis=1, inplace=True)

__Dividindo o conjunto de dados__

In [ ]:
objetivo = dados_ohe['Exited']
carateristicas = dados_ohe.drop('Exited', axis=1)
carateristicas_train, carateristicas_valid, objetivo_train, objetivo_valid = train_test_split( carateristicas, 
                                          objetivo, test_size=.25, random_state=54321)

__Equalizando as escalas das carateristicas__

In [ ]:
numericos = [ 'CreditScore', 'Age', 'Tenure', 'Balance', 'NumOfProducts', 'EstimatedSalary']
scaler = StandardScaler()
scaler.fit(carateristicas_train[numericos])
carateristicas_train[numericos] = scaler.transform(carateristicas_train[numericos])

carateristicas_valid[numericos] = scaler.transform(carateristicas_valid[numericos])

In [ ]:
carateristicas_train = carateristicas_train.drop('Surname', axis=1)


In [ ]:
carateristicas_valid = carateristicas_valid.drop('Surname', axis=1)

## Equilibrio de Classes

__Examinando o equilibrio das classes__

In [ ]:
class_counts = dados['Exited'].value_counts()
percentage_exited = class_counts[1] / len(dados) * 100
percentage_not_exited = class_counts[0] / len(dados) * 100

print(f"Quantidade de clientes que saíram do banco: {class_counts[1]} ({percentage_exited:.2f}%)")
print(f"Quantidade de clientes que não saíram do banco: {class_counts[0]} ({percentage_not_exited:.2f}%)")

__Treinando o modelo sem considerar o equilibrio entre as classes__

In [ ]:
#RandomForestClassifier
rf_classifier_modelo = RandomForestClassifier(n_estimators=100, random_state=54321)
rf_classifier_modelo.fit(carateristicas_train, objetivo_train)

__Testando o modelo treinado__

In [ ]:
predicoes = rf_classifier_modelo.predict(carateristicas_valid)
acuracia = accuracy_score(objetivo_valid, predicoes)
print(f"Acuracia: {acuracia:.2f}")
f1 = f1_score(objetivo_valid, predicoes, average='weighted')
print(f"F1-score: {f1:.2f}")

probabilities_valid = rf_classifier_modelo.predict_proba(carateristicas_valid)
probabilities_one_valid = probabilities_valid[:, 1]

auc_roc = roc_auc_score(objetivo_valid, probabilities_one_valid)
print(f"Auc-Roc: {auc_roc:.2f}")


### Breves Conclusões

O resultado obtido é bastante promissor, com uma acurácia de 87% e um F1-score de 86%. A alta acurácia indica que o modelo possui uma taxa significativa de previsões corretas em relação ao total de amostras. Além disso, o F1-score, que é uma métrica que leva em consideração tanto a precisão quanto a sensividade, também apresenta um valor elevado, indicando que o modelo está equilibrando bem as previsões para ambas as classes.

Esses resultados sugerem que o modelo é capaz de fazer previsões precisas e confiáveis para identificar clientes que estão prestes a deixar o banco. Isso pode ser de grande valor para o Beta Bank, pois permitirá que eles tomem medidas proativas para reter clientes em risco, reduzindo o churn e melhorando a satisfação do cliente.

No entanto, é importante lembrar que a métrica de acurácia por si só pode ser enganosa em problemas de desequilíbrio de classes. Portanto, a avaliação do F1-score também é essencial, especialmente em cenários onde uma classe é muito menor do que a outra. Esses resultados demonstram o sucesso das técnicas empregadas para lidar com o desequilíbrio de classes e a eficácia do modelo desenvolvido. Entretanto, é sempre recomendável continuar monitorando e aprimorando o modelo à medida que novos dados se tornam disponíveis para garantir a manutenção de um desempenho consistente e confiável.

## Melhorando o Modelo

__Executando a Superamostragem__

In [ ]:
#Funcao para executar a superamostragem e retorna o conjunto com as classes equilibradas
def elevar_classes(carateristicas, objetivo, repeticao):
    
    carateristicas_zeros = carateristicas[objetivo == 0]
    carateristicas_uns = carateristicas[objetivo ==1]
    objetivo_zeros = objetivo[objetivo == 0]
    objetivo_uns = objetivo[objetivo == 1]
    
    carateristicas_elevadas = pd.concat([carateristicas_zeros] + [carateristicas_uns] * repeticao)
    objetivo_elevado = pd.concat([objetivo_zeros] + [objetivo_uns] * repeticao)
    
    carateristicas_elevadas, objetivo_elevado = shuffle( carateristicas_elevadas, objetivo_elevado, random_state = 54321)
    
    return carateristicas_elevadas, objetivo_elevado

carateristicas_elevadas, objetivo_elevado = elevar_classes(carateristicas_train, objetivo_train, 20)

__Executando a Subamostragem__

In [ ]:
#Funcao para executar a subamostragem e retorna o conjunto com as classes ponderadas para baixo
def baixar_classes(carateristicas, objetivo, fracao):
    carateristicas_zeros = carateristicas[objetivo == 0]
    carateristicas_uns = carateristicas[objetivo == 1]
    objetivo_zeros = objetivo[objetivo == 0]
    objetivo_uns = objetivo[objetivo == 1]
    
    carateristicas_abaixadas = pd.concat( [carateristicas_zeros.sample(frac=fracao, random_state=54321)] + 
                                        [carateristicas_uns])
    
    objetivo_abaixado = pd.concat( [objetivo_zeros.sample(frac=fracao, random_state=54321)] +
                                 [objetivo_uns])
    
    return carateristicas_abaixadas, objetivo_abaixado

carateristicas_abaixadas, objetivo_abaixado = baixar_classes( carateristicas_train, objetivo_train, 0.1)

__Unificando os dois conjuntos em um unico conjunto com classes equilibradas__

In [ ]:
carateristicas_equilibradas = pd.concat([carateristicas_elevadas, carateristicas_abaixadas], axis=0)
objetivo_equilibrado = pd.concat([objetivo_elevado, objetivo_abaixado], axis=0)

__Embaralhando os dados para garantir aleatoriedade__

In [ ]:
carateristicas_equilibradas = carateristicas_equilibradas.sample(frac=1, random_state=54321).reset_index(drop=True)
objetivo_equilibrado = objetivo_equilibrado.sample(frac=1, random_state=54321).reset_index(drop=True)

__Eliminando a coluna Surname__

In [ ]:
carateristicas_equilibradas = carateristicas_equilibradas.drop('Surname', axis=1)

In [ ]:
rf_classifier_modelo.fit(carateristicas_equilibradas, objetivo_equilibrado)

### Execução de validação cruzada para encontrar o melhor modelo e os melhores hiperparametros

__Definição dos hiperparametros a serem otimizados__

In [ ]:
modelo_melhorado = RandomForestClassifier(random_state=54321)


param_grid = {
    'n_estimators' : [50, 100, 150],
    'max_depth' : [None, 5, 10],
    'min_samples_split': [2, 5, 10]
}




__Realização de busca exaustiva pelos hiperparametros usando validação cruzada__

In [ ]:
grid_search = GridSearchCV(modelo_melhorado, param_grid, cv=5, scoring='f1')
grid_search.fit(carateristicas_equilibradas, objetivo_equilibrado)

__Obtendo os melhores hiperparametros e o melhor modelo__

In [ ]:
best_params = grid_search.best_params_
best_model = grid_search.best_estimator_

## Teste Final

__Testando o modelo__

In [ ]:
predicoes_melhorada = best_model.predict(carateristicas_valid)
f1_melhorado = f1_score(objetivo_valid, predicoes_melhorada)

probabilities_valid_melhorado = best_model.predict_proba(carateristicas_valid)
probabilities_one_valid_melhorado = probabilities_valid_melhorado[:, 1]

auc_roc = roc_auc_score(objetivo_valid, probabilities_one_valid_melhorado)
print(f"Auc-Roc: {auc_roc:.2f}")
print(f"F1:{f1_melhorado:.2f}")

__Breves conclusões__

Após aplicar as técnicas de oversampling, undersampling e a validação cruzada, obtivemos resultados promissores, mas com algumas observações importantes. O modelo aprimorado alcançou uma AUC-ROC de 85%, o que indica que ele é capaz de fazer distinções significativas entre as classes e possui uma capacidade satisfatória de classificação.

No entanto, o F1-score obtido de 56% merece atenção, pois indica que o modelo pode não estar alcançando um bom equilíbrio entre a precisão e o recall, especialmente para a classe minoritária. Isso pode ser atribuído a algumas dificuldades inerentes ao tratamento do desequilíbrio de classes, pois é um desafio garantir que o modelo não esteja tendencioso em direção à classe majoritária.

É importante notar que os resultados são uma combinação de esforços para melhorar o modelo, incluindo a escolha adequada de hiperparâmetros e técnicas de amostragem. É possível que ajustes adicionais nas configurações do modelo ou a exploração de outras técnicas de balanceamento de classes possam melhorar ainda mais o desempenho.

Em conclusão, embora tenhamos obtido melhorias significativas no desempenho do modelo após aplicar oversampling, undersampling e validação cruzada, ainda há espaço para otimizações futuras. A abordagem adotada mostrou-se valiosa para mitigar o desequilíbrio de classes, mas a busca por melhores soluções é uma jornada contínua para alcançar uma maior capacidade preditiva e uma generalização ainda mais sólida.






# Conclusão Geral


O objetivo foi construir um modelo capaz de prever se um cliente do Beta Bank deixaria o banco em breve. O conjunto de dados continha diversas características dos clientes, incluindo informações demográficas, pontuação de crédito, saldos de conta e comportamento passado.

Inicialmente, o conjunto de dados foi preparado, tratando valores ausentes e transformando variáveis categóricas em numéricas. Em seguida, o equilíbrio das classes foi investigado, e foi constatado um desequilíbrio significativo, com um número maior de clientes que não saíram do banco em comparação com aqueles que saíram.

No primeiro modelo, o desequilíbrio de classe não foi tratado, e o modelo alcançou uma acurácia alta de 87%. No entanto, o resultado foi enganoso, pois o F1-score foi de 86%, o que indicou que o modelo não estava equilibrando adequadamente as previsões entre as classes.

Para melhorar o modelo, foram aplicadas técnicas de undersampling e oversampling para equilibrar as classes. A validação cruzada foi utilizada para encontrar os melhores hiperparâmetros e aprimorar a generalização do modelo.

No modelo final, obteve-se uma AUC-ROC de 85%, demonstrando uma capacidade aceitável de discriminação entre as classes. Entretanto, o F1-score alcançou apenas 56%, indicando dificuldades em equilibrar precisão e recall.

Surpreendentemente, o modelo inicial apresentou resultados superiores em relação ao modelo final. Esse cenário pode ser explicado pelo risco de overfitting no modelo final, resultado do balanceamento excessivo das classes ou da complexidade inadequada do algoritmo utilizado.

Em conclusão, apesar de melhorar a AUC-ROC, o modelo final não apresentou um desempenho equilibrado entre as classes, tornando-o menos eficaz para identificar corretamente os clientes em risco de sair do banco. Esses resultados ressaltam a importância de explorar diferentes técnicas, ajustar parâmetros e considerar cuidadosamente o desequilíbrio de classes para desenvolver um modelo confiável e de alto desempenho. A busca pela melhoria contínua e a compreensão profunda dos dados são essenciais para o sucesso em projetos de ciência de dados.